# Wetlands location preparation

In [1]:
import json

import geopandas as gpd
import pandas as pd


In [2]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"

## Global (Sahel) boundaries

In [3]:
sahel_boundary = gpd.read_file(f'{gcs_path}/sahel_boundary.gpkg', layer='Sahel-zone - extended - dissolved')
sahel_boundary['name'] = 'Global (Sahel)'
sahel_boundary['type'] = 'global'
sahel_boundary['id'] = 'global_sahel'
sahel_boundary['code'] = 'global_sahel'
sahel_boundary['bbox'] = sahel_boundary['geometry'].apply(lambda x: x.bounds)
sahel_boundary['bbox'] = sahel_boundary['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])

sahel_boundary = sahel_boundary[['id', 'name', 'type', 'code', 'bbox', 'geometry']]
sahel_boundary


,id,name,type,code,bbox,geometry
0,global_sahel,Global (Sahel),global,global_sahel,"[-18.46394608680282, -4.900325000877443, 48.01...","MULTIPOLYGON (((42.94782 10.99465, 42.97027 10..."


## Country boundaries

In [4]:
gadm_boundaries = gpd.read_file(f'{gcs_path}/sahel_countries_gadm.geojson')
gadm_boundaries.head(3)

,fid,GID_0,COUNTRY,geometry
0,201,SEN,Senegal,"MULTIPOLYGON (((-15.95338 12.44265, -15.98577 ..."
1,90,GMB,Gambia,"MULTIPOLYGON (((-16.69071 13.16528, -16.69050 ..."
2,125,KEN,Kenya,"MULTIPOLYGON (((38.76080 -4.35965, 38.67352 -4..."


In [5]:
# Load country boundaries gpkg from GCS bucket
osm_file = 'OSM_admin1_Sahel.gpkg'
country_boundaries = gpd.read_file(f"{gcs_path}/{osm_file}", layer='OSM_admin1_Sahel')
country_boundaries['name_en'] = country_boundaries['name_en'].str.replace('The Gambia', 'Gambia')
country_boundaries.head(3)

,osm_id,name,name_en,boundary,admin_level,admin_centre_node_id,admin_centre_node_lat,admin_centre_node_lng,label_node_id,label_node_lat,label_node_lng,layer,path,geometry
0,-192774,Gambia,Gambia,administrative,2,249167555,13.45535,-16.575646,424297579,13.470062,-15.4900464,OSMB-304a0d72313914c657dd9cedeb1cd60a8327eb16,C:/Users/Lammert/Downloads/OSMB-304a0d72313914...,"MULTIPOLYGON (((-17.02238 13.38599, -17.02238 ..."
1,-192797,Elemi Triangle مثلث إليمى,Ilemi Triangle,administrative,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OSMB-78165b1ebda0c70bdd37158924af4ab2d72b0f35,C:/Users/Lammert/OneDrive - Wetlands Internati...,"MULTIPOLYGON (((34.38750 4.61000, 34.39417 4.6..."
2,-192763,موريتانيا,Mauritania,administrative,2,249166495,18.0792379,-15.9780071,424316493,20.2540382,-9.2399263,OSMB-ede040002cf53abbaefb675e6bb5caf8cd357bb4,C:/Users/Lammert/OneDrive - Wetlands Internati...,"MULTIPOLYGON (((-17.23810 20.66622, -17.23676 ..."


In [6]:
sahel_countries = pd.merge(gadm_boundaries[['GID_0', 'COUNTRY']], country_boundaries, left_on='COUNTRY', right_on='name_en', how='inner')
sahel_countries = sahel_countries[['GID_0', 'COUNTRY', 'geometry']]
sahel_countries = gpd.GeoDataFrame(sahel_countries, geometry='geometry', crs='EPSG:4326')
sahel_countries.head(3)

,GID_0,COUNTRY,geometry
0,SEN,Senegal,"MULTIPOLYGON (((-17.74987 14.74329, -17.74984 ..."
1,GMB,Gambia,"MULTIPOLYGON (((-17.02238 13.38599, -17.02238 ..."
2,KEN,Kenya,"MULTIPOLYGON (((33.90969 0.09973, 33.90984 0.0..."


In [7]:
print(f'GADM countries: {len(gadm_boundaries)}')
print(f'Original Sahel boundaries: {len(country_boundaries)}')
print(f'Combined Sahel countries: {len(sahel_countries)}')

GADM countries: 37
Original Sahel boundaries: 27
Combined Sahel countries: 26


In [8]:
#Missing geometry in the combined table
country_boundaries[~country_boundaries['name_en'].isin(gadm_boundaries['COUNTRY'])]

,osm_id,name,name_en,boundary,admin_level,admin_centre_node_id,admin_centre_node_lat,admin_centre_node_lng,label_node_id,label_node_lat,label_node_lng,layer,path,geometry
1,-192797,Elemi Triangle مثلث إليمى,Ilemi Triangle,administrative,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OSMB-78165b1ebda0c70bdd37158924af4ab2d72b0f35,C:/Users/Lammert/OneDrive - Wetlands Internati...,"MULTIPOLYGON (((34.38750 4.61000, 34.39417 4.6..."


In [9]:
sahel_countries.rename(columns={'GID_0': 'code', 'COUNTRY': 'name'}, inplace=True)
sahel_countries['type'] = 'admin_region'
sahel_countries['id'] = sahel_countries['code'].apply(lambda x: f"adminregion_{x.lower()}")
# get bounding box
sahel_countries['bbox'] = sahel_countries['geometry'].apply(lambda x: x.bounds)
sahel_countries['bbox'] = sahel_countries['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])


sahel_countries = sahel_countries[['id', 'name', 'type', 'code', 'bbox', 'geometry']]
sahel_countries.sample(5)

,id,name,type,code,bbox,geometry
4,adminregion_mrt,Mauritania,admin_region,MRT,"[-17.2380959, 14.7209909, -4.8333344, 27.314942]","MULTIPOLYGON (((-17.23810 20.66622, -17.23676 ..."
17,adminregion_tcd,Chad,admin_region,TCD,"[13.47348, 7.44107, 24.0, 23.4507957]","MULTIPOLYGON (((13.47348 14.44282, 13.47622 14..."
6,adminregion_gnb,Guinea-Bissau,admin_region,GNB,"[-16.897501, 10.6514215, -13.6348777, 12.6862384]","MULTIPOLYGON (((-16.89750 12.24027, -16.87912 ..."
0,adminregion_sen,Senegal,admin_region,SEN,"[-17.7498686, 12.2402664, -11.3459503, 16.6919...","MULTIPOLYGON (((-17.74987 14.74329, -17.74984 ..."
21,adminregion_sdn,Sudan,admin_region,SDN,"[21.8143904, 8.685278, 39.0576252, 22.224918]","MULTIPOLYGON (((21.81439 12.80858, 21.81470 12..."


## Hydrobasins

In [10]:
hydrobasins = gpd.read_file(f'{gcs_path}/sahel_hydrobasins.geojson')
hydrobasins.rename(columns={'HYBAS_ID': 'code', 'NAME': 'name'}, inplace=True)
display(hydrobasins.head(1))

hydrobasins_names = gpd.read_file(f'{gcs_path}/river%20basins%20clipped%20sahel_names.gpkg', layer='river_basins_clipped_sahel_withNames')
hydrobasins_names = hydrobasins_names[['HYBAS_ID', 'Name']]
hydrobasins_names = hydrobasins_names[hydrobasins_names['Name'] != '0']
display(hydrobasins_names.head(1))

,fid,code,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,COAST,ORDER,SORT,geometry
0,1,1030000010,0,1030000010,1030000010,0.0,0.0,236343.2,236343.2,111,0,1,0,1,"MULTIPOLYGON (((40.28333 14.92917, 40.28425 14..."


,HYBAS_ID,Name
0,1030000010,Red Sea


In [11]:
hydrobasins_names

,HYBAS_ID,Name
0,1030000010,Red Sea
1,1030003990,Horn of Africa
2,1030008100,Juba-Shibeli
3,1030008110,Tana
4,1030020040,Congo
6,1030022420,Niger
7,1030022430,Ouémé
8,1030023300,Volta
9,1030023310,West Coast
10,1030027410,Senegal


In [12]:
sahel_hydrobasins = pd.merge(hydrobasins, hydrobasins_names, left_on='code', right_on='HYBAS_ID', how='left')
sahel_hydrobasins.dropna(inplace=True)
sahel_hydrobasins = sahel_hydrobasins[['Name', 'code', 'geometry']]
sahel_hydrobasins.rename(columns={'Name': 'name'}, inplace=True)
sahel_hydrobasins['name'] = sahel_hydrobasins['name'] + ' Basin'
sahel_hydrobasins['type'] = 'hydro_basin'
sahel_hydrobasins['id'] = sahel_hydrobasins['name'].apply(lambda x: f"hydrobasin_{x.lower().replace(' ', '_').replace('-', '_')}")
sahel_hydrobasins['bbox'] = sahel_hydrobasins['geometry'].apply(lambda x: x.bounds)
sahel_hydrobasins['bbox'] = sahel_hydrobasins['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])
sahel_hydrobasins = sahel_hydrobasins[['id', 'name', 'type', 'code', 'bbox', 'geometry']]
sahel_hydrobasins = sahel_hydrobasins[sahel_hydrobasins['name'] != '0']
print(len(sahel_hydrobasins))
sahel_hydrobasins.head(3)

17


,id,name,type,code,bbox,geometry
0,hydrobasin_red_sea_basin,Red Sea Basin,hydro_basin,1030000010,"[31.64526604546444, 14.72048000759551, 40.8797...","MULTIPOLYGON (((40.28333 14.92917, 40.28425 14..."
1,hydrobasin_horn_of_africa_basin,Horn of Africa Basin,hydro_basin,1030003990,"[40.603684658474414, -0.258333333333303, 51.41...","MULTIPOLYGON (((44.04039 1.09936, 44.03408 1.1..."
2,hydrobasin_juba_shibeli_basin,Juba-Shibeli Basin,hydro_basin,1030008100,"[36.3125, -0.429166666666638, 46.1541666666666...","MULTIPOLYGON (((36.74167 4.06667, 36.74131 4.0..."


## Protected Areas

In [13]:
analysis_data_folder = 'Analysis_Data_insight_Wetlands_Inventory'
file_path = f"{gcs_path}/WDPA_areas/WDPA_names%20updated.shp"
all_wdpa = gpd.read_file(file_path)
all_wdpa = all_wdpa.to_crs(epsg=4326)
all_wdpa['type'] = 'wdpa'
all_wdpa = all_wdpa[['NAME', 'fid', 'ramsarid', 'type', 'geometry']]
all_wdpa.rename(columns={'NAME': 'name'}, inplace=True)
all_wdpa.head(3)
#TODO: get corrected names from WDPA dataset

,name,fid,ramsarid,type,geometry
0,Tana River Delta Ramsar Site,1,2082,wdpa,"POLYGON ((40.13725 -2.60449, 40.11734 -2.52755..."
1,Lake Bogoria,"2,410",1097,wdpa,"POLYGON ((36.06645 0.34045, 36.06729 0.34143, ..."
2,Lake Baringo,3,1159,wdpa,"POLYGON ((36.00937 0.61575, 36.01053 0.61754, ..."


In [14]:
print(f'Total protected areas: {len(all_wdpa)}')
print(f'Total unique protected area names: {all_wdpa["name"].nunique()}')
print(f'Total unique Ramsar IDs: {all_wdpa["ramsarid"].nunique()}')
print(f'Total unique fid: {all_wdpa["fid"].nunique()}')
print(f'Areas without fid: {len(all_wdpa[all_wdpa["fid"].isna()])}')

Total protected areas: 2024
Total unique protected area names: 2024
Total unique Ramsar IDs: 65
Total unique fid: 1968
Areas without fid: 0


In [15]:
f = pd.DataFrame(all_wdpa['fid'].value_counts())
len(f[f['count'] > 1])

56

In [16]:
wdpa_final = all_wdpa[['name', 'type', 'fid', 'geometry']].copy()
#wdpa_final.rename(columns={'fid': 'code'}, inplace=True)
wdpa_final['id'] = wdpa_final.index.to_series().apply(lambda x: f"wdpa_{x}")
#merge id and name for code
wdpa_final['code'] = wdpa_final.apply(lambda row: f"{row['id']}_{row['name'].lower().replace(' ', '_').replace('-', '_')}", axis=1)
wdpa_final['bbox'] = wdpa_final['geometry'].apply(lambda geom: geom.bounds)
wdpa_final['bbox'] = wdpa_final['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])
wdpa_final = wdpa_final[['id', 'name', 'type', 'code', 'bbox', 'geometry']]
wdpa_final.head(10)



,id,name,type,code,bbox,geometry
0,wdpa_0,Tana River Delta Ramsar Site,wdpa,wdpa_0_tana_river_delta_ramsar_site,"[40.11119584, -2.73808143, 40.56739413, -2.260...","POLYGON ((40.13725 -2.60449, 40.11734 -2.52755..."
1,wdpa_1,Lake Bogoria,wdpa,wdpa_1_lake_bogoria,"[36.057949879000034, 0.169972509000047, 36.142...","POLYGON ((36.06645 0.34045, 36.06729 0.34143, ..."
2,wdpa_2,Lake Baringo,wdpa,wdpa_2_lake_baringo,"[36.0070767, 0.45247296, 36.16329205, 0.74772578]","POLYGON ((36.00937 0.61575, 36.01053 0.61754, ..."
3,wdpa_3,Lake Naivasha,wdpa,wdpa_3_lake_naivasha,"[36.25276154, -0.83770029, 36.44270729, -0.595...","POLYGON ((36.30438 -0.69476, 36.30609 -0.69190..."
4,wdpa_4,Lake Elmenteita,wdpa,wdpa_4_lake_elmenteita,"[36.20157704, -0.55815977, 36.29881022, -0.381...","POLYGON ((36.25939 -0.53468, 36.24783 -0.55676..."
5,wdpa_5,Lake Nakuru,wdpa,wdpa_5_lake_nakuru,"[36.03000825, -0.4983706459999553, 36.15671117...","POLYGON ((36.13187 -0.31086, 36.13189 -0.31152..."
6,wdpa_6,Zone humide du moyen Niger,wdpa,wdpa_6_zone_humide_du_moyen_niger,"[3.00013375, 12.00013828, 3.30723834, 12.33365...","POLYGON ((3.00020 12.31031, 3.00021 12.31417, ..."
7,wdpa_7,Dallol Maouri,wdpa,wdpa_7_dallol_maouri,"[3.34362388, 11.69602299, 3.84224582, 12.93610...","POLYGON ((3.42879 12.17817, 3.42276 12.18826, ..."
8,wdpa_8,La Mare de Tabalak,wdpa,wdpa_8_la_mare_de_tabalak,"[5.56180263, 14.89432144, 6.15121191, 15.27905...","POLYGON ((5.64080 15.03309, 5.64148 15.03898, ..."
9,wdpa_9,Gueltas et Oasis de l'Aïr,wdpa,wdpa_9_gueltas_et_oasis_de_l'aïr,"[7.8665819, 17.33377778, 9.99060117, 20.58517194]","POLYGON ((8.29110 19.41563, 8.24893 19.45780, ..."


## Ecoregions

## Combine locations

In [17]:
locations = pd.concat([sahel_boundary, sahel_countries, sahel_hydrobasins, wdpa_final], ignore_index=True)
locations.sample(4)

,id,name,type,code,bbox,geometry
494,wdpa_450,Diani Chale Marine,wdpa,wdpa_450_diani_chale_marine,"[39.52353854100005, -4.463139043999945, 39.631...","POLYGON ((39.63180 -4.25446, 39.61872 -4.28796..."
484,wdpa_440,Arawale,wdpa,wdpa_440_arawale,"[40.02445865200008, -1.5719009829999777, 40.34...","POLYGON ((40.13735 -1.29315, 40.14068 -1.29674..."
1453,wdpa_1409,Ribako,wdpa,wdpa_1409_ribako,"[7.767180000000053, 10.539690000000064, 7.9187...","POLYGON ((7.84365 10.66270, 7.84701 10.66173, ..."
1267,wdpa_1223,Shimfida,wdpa,wdpa_1223_shimfida,"[7.09773000000007, 12.81606000000005, 7.120410...","POLYGON ((7.12041 12.82152, 7.11867 12.82070, ..."


In [18]:
locations_simplified = locations.copy()
locations_simplified['geometry'] = locations_simplified['geometry'].simplify(tolerance=0.005, preserve_topology=True)
locations_simplified.head(3)

,id,name,type,code,bbox,geometry
0,global_sahel,Global (Sahel),global,global_sahel,"[-18.46394608680282, -4.900325000877443, 48.01...","POLYGON ((42.94782 10.99465, 42.97027 10.99147..."
1,adminregion_sen,Senegal,admin_region,SEN,"[-17.7498686, 12.2402664, -11.3459503, 16.6919...","POLYGON ((-17.74987 14.74329, -17.74334 14.693..."
2,adminregion_gmb,Gambia,admin_region,GMB,"[-17.0223778, 13.0558333, -13.797778, 13.8253137]","POLYGON ((-17.02238 13.38599, -17.01023 13.249..."


In [19]:
# display as json with indentation
locations_json = locations_simplified.to_json(indent=2)
locations_json = json.loads(locations_json)
#locations_json['features'][0]

In [20]:
locations_simplified.drop(columns='bbox', inplace=True)
locations_simplified.to_file('../data/processed/locations_simplified.geojson', driver='GeoJSON')

In [21]:
locations_list = []
for f in locations_json['features']:
    loc = f['properties']
    loc['geometry'] = f['geometry']
    locations_list.append(loc)

#locations_list

In [22]:
len(locations_list)

2068

In [23]:
locations_simplified['type'].value_counts()

type
wdpa            2024
admin_region      26
hydro_basin       17
global             1
Name: count, dtype: int64

In [24]:
# save list as json
with open('../../app-initial-data/locations.json', 'w') as f:
    json.dump(locations_list, f, indent=2)

### Ramsar sites

In [20]:
ramsar_poly = gpd.read_file(f'{gcs_path}/ramsar_polygons_sahel_dissolved.gpkg', layer='dissolved')
ramsar_poly['type'] = 'ramsar_polygons'
ramsar_poly.head(3)

,v_idris,ramsarid,officialname,iso3,country_en,area_off,geometry,type
0,38563073,2082,Tana River Delta Ramsar Site,KEN,Kenya,163600.0,"MULTIPOLYGON (((40.13725 -2.60449, 40.11734 -2...",ramsar_polygons
1,48385794,1097,Lake Bogoria,KEN,Kenya,10700.0,"MULTIPOLYGON (((36.13286 0.19536, 36.13032 0.1...",ramsar_polygons
2,48392137,1159,Lake Baringo,KEN,Kenya,31469.0,"MULTIPOLYGON (((36.00937 0.61575, 36.01053 0.6...",ramsar_polygons


In [21]:
ramsar_points = gpd.read_file(f'{gcs_path}/ramsar_points_sahel.gpkg', layer='features_centroid')
ramsar_points['type'] = 'ramsar_centroids'
ramsar_points.head(3)

,v_idris,geometry,type
0,38563073,POINT (40.28654 -2.49181),ramsar_centroids
1,2086,POINT (40.28333 -2.45000),ramsar_centroids
2,38271242,POINT (40.28333 -2.45000),ramsar_centroids


In [23]:
ramsar_combined = pd.concat([ramsar_poly, ramsar_points], ignore_index=True)
ramsar_combined = ramsar_combined[['v_idris','type', 'geometry']]
ramsar_combined['type'].value_counts()

type
ramsar_centroids    237
ramsar_polygons      73
Name: count, dtype: int64

In [24]:
ramsar_combined.to_file('../data/processed/ramsar_sites.geojson', driver='GeoJSON')